# SVD++ / MovieLens 1M — обучение в Google Colab

Ноутбук запускает Python-файлы из папки `svdpp/` общего GitHub-репозитория.
Формула модели — в `src/model.py`, цикл обучения — в `src/training.py`.
Данные скачиваются и проверяются автоматически перед обучением.

Перед запуском выберите **Среда выполнения → Сменить среду выполнения → GPU**.
Запускайте ячейки по порядку кнопкой ▶ или `Shift+Enter`.


## 1. Подключить Google Drive

Разрешите подключение аккаунта. Копия исходников, контрольные точки и результаты
будут находиться в `Мой диск/SVDpp_MovieLens1M`. Данные читаются с диска Colab.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/SVDpp_MovieLens1M")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DRIVE_DIR / "ai360_recsys_source.zip"
print("Папка результатов:", DRIVE_DIR)


## 2. Получить проект с GitHub

При первом запуске исходники скачиваются из GitHub и сохраняются на Drive.
Следующие сессии используют ту же копию кода для продолжения обучения.
Для нового эксперимента с обновлённым кодом задайте другое имя `ZIP_PATH`.


In [ ]:
import shutil
import zipfile
import os
import subprocess

REPOSITORY = Path("/content/ai360-recsys-2026")
PROJECT = REPOSITORY / "svdpp"
if not (PROJECT / "train.py").exists():
    if ZIP_PATH.exists():
        with zipfile.ZipFile(ZIP_PATH) as archive:
            archive.extractall(REPOSITORY)
    else:
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/polushkin-ayu/ai360-recsys-2026.git",
                        str(REPOSITORY)], check=True)
        if not (PROJECT / "train.py").exists():
            raise RuntimeError("Новая папка svdpp ещё не опубликована в main.")
        shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", root_dir=REPOSITORY)
os.chdir(PROJECT)
print("Проект:", PROJECT)
print("Модель:", PROJECT / "src/model.py")
print("Обучение:", PROJECT / "src/training.py")


## 3. Сохранение результатов на Drive

Папки результатов связываются с Drive. Данные остаются в `/content`.
Одиночный `train.py` сохраняет контрольную точку после каждой эпохи;
быстрый режим — с интервалом из конфигурации. После разрыва обучение
продолжается с последней сохранённой контрольной точки.


In [ ]:
for relative in ("results/single", "results/svdpp-ml1m", "results/svdpp-ml1m-fast", "artifacts"):
    local = PROJECT / relative
    saved = DRIVE_DIR / relative
    saved.mkdir(parents=True, exist_ok=True)
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        if local.resolve() != saved.resolve():
            raise ValueError("Папка уже связана с другим местом: " + str(local))
    elif local.exists():
        if any(local.iterdir()):
            raise ValueError("В локальной папке уже есть результаты: " + str(local))
        local.rmdir()
        local.symlink_to(saved, target_is_directory=True)
    else:
        local.symlink_to(saved, target_is_directory=True)
print("Контрольные точки и результаты будут сохраняться на Google Drive.")

## 4. Установить зависимости

Используем PyTorch, установленный в Colab: установка CPU-версии здесь не нужна.


In [ ]:
requirements_path = str(PROJECT / "requirements.txt")
%pip install -q -r $requirements_path


## 5. Проверить GPU

Если CUDA недоступна, включите GPU в настройках и подключитесь к новой среде.


In [ ]:
import torch
import numpy as np
import pandas as pd
import sys

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__, "Pandas:", pd.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Включите GPU: Runtime → Change runtime type → GPU.")
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

## 6. Проверить данные

Используется весь train: **700 146 оценок**. Валидация: **150 002 оценки**.
Для теста подготовлены **149 990 оценок**, но этот запуск его не оценивает.
`audit` сверяет контрольные суммы; целевые оценки test не загружаются.


In [ ]:
import subprocess

subprocess.run([sys.executable, "prepare_data.py"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "experiment.py", "audit", "--device", DEVICE],
               cwd=PROJECT, check=True)


## 7. Настроить одну модель

Это пример стартовых параметров, а не результат полного подбора.
`K` — длина эмбеддингов; `LR` — скорость обучения; `REG` — lambda в
`MSE + lambda/N * L2`; `EPOCHS` — общее число эпох; `SEED` — начальная случайность.
`RUN_NAME` — отдельная папка для этого опыта. Если меняете параметры, меняйте
имя папки; для продолжения оставьте параметры и имя прежними.


In [ ]:
K = 20
LR = 0.003
REG = 1.0
EPOCHS = 100
SEED = 11
RUN_NAME = "k20_lr0003_reg1_seed11"
RUN_DIR = PROJECT / "results/single" / RUN_NAME

command = [sys.executable, "-u", "train.py",
           "--k", str(K), "--lr", str(LR), "--reg", str(REG),
           "--epochs", str(EPOCHS), "--seed", str(SEED), "--device", DEVICE,
           "--output", str(RUN_DIR)]
print(" ".join(command))

## 8. Обучить

В логах `train` и `val` — RMSE после эпохи, `best` — минимальный validation RMSE.
Лучшая эпоха выбирается по validation. Повторный запуск этой ячейки продолжает
обучение из `last.pt`. Увеличить общий лимит можно в `EPOCHS` предыдущей ячейки.


In [ ]:
subprocess.run(command, cwd=PROJECT, check=True)

## 9. Посмотреть качество и график

После завершения `model.pt` содержит лучшую модель по validation, `last.pt` —
состояние для продолжения. При early stopping они могут отличаться.


In [ ]:
import json
from IPython.display import display, Image

metrics = json.loads((RUN_DIR / "metrics.json").read_text())
print(json.dumps(metrics, indent=2, ensure_ascii=False))
display(Image(filename=str(RUN_DIR / "learning_curve.png")))
display(Image(filename=str(RUN_DIR / "train_loss.png")))
display(pd.read_csv(RUN_DIR / "history.csv").tail())
print("Результаты на Drive:", DRIVE_DIR / "results/single" / RUN_NAME)

## 10. Скачать результаты

Они уже сохранены на Drive. Эта ячейка дополнительно создаёт ZIP для ноутбука.


In [ ]:
from google.colab import files

result_zip = shutil.make_archive("/content/SVDpp_" + RUN_NAME + "_results",
                                 "zip", root_dir=RUN_DIR)
files.download(result_zip)

## Как продолжить после разрыва Colab

Подключите GPU, повторите ячейки 1–7, оставив прежними `K`, `LR`, `REG`, `SEED`
и `RUN_NAME`. Запустите ячейку 8. Исходники и checkpoint будут взяты с Drive;
в логах появится `resume epoch ...`. `EPOCHS=100` означает довести обучение
до 100-й эпохи. Если early stopping уже сработал, увеличение лимита эпох
не добавит новые эпохи. Смена CPU/GPU может слегка изменить численные результаты.

## Полный подбор и график RMSE(k): необязательная длительная серия

Для каждого `k` нужно отдельно подобрать `lr` и `lambda` по validation.
`tune` автоматически создаёт тепловые карты и кривые лучших кандидатов в `tuning_plots/`.
Перерисовка без обучения: `!python -u experiment.py plot-tuning --device cuda`.
`fit` сохраняет loss по эпохам каждого финального запуска и обзор `final_loss_curves.png`.
`plot` строит validation RMSE от k; после этапа `test` добавляется test-кривая.
По умолчанию подбор двухэтапный: все 8 пар для каждого k сравниваются на seed 11
за 100 эпох; лучшая lambda для каждого lr подтверждается на seeds 11, 22, 33
до 500 эпох, минимум 100. Seed 11 продолжается из сохранённого состояния.
Это 144 предварительных запуска и 108 подтверждений (36 продолжаются), затем
90 финальных повторов. Test в отборе не используется.
`screening_heatmap_k<k>.png` показывает предварительные метрики и оставленные пары,
`heatmap_k<k>.png` — среднее и SD подтверждений на трёх seeds.
Можно увеличить `candidates_per_learning_rate` или `screening_epochs` в config.
Полный перебор: `tuning_strategy: {"mode": "grid"}` в копии config с новыми
output/checkpoint-папками.
Сначала можно запустить только подбор для `k=2`, чтобы измерить длительность.
Простое перебирание `K` при одной паре параметров даёт предварительный график.

Ниже команды полного протокола. Выполняйте их отдельными ячейками по мере
завершения предыдущих этапов; `--k 2` ограничивает только этап tune.

```python
!python -u experiment.py benchmark --device cuda --k 2 200
!python -u experiment.py tune --device cuda --k 2
# Затем завершить подбор для всех k:
!python -u experiment.py tune --device cuda
!python -u experiment.py diagnose --device cuda
!python -u experiment.py fit --device cuda
!python -u experiment.py diagnose --device cuda
# Проверить диагностику до фиксации и доступа к test:
!python -u experiment.py freeze --device cuda
!python -u experiment.py test --device cuda
!python -u experiment.py plot --device cuda
```

График: `results/svdpp-ml1m/rmse_vs_k.png`.
Полная серия сохраняет строгий fingerprint окружения, поэтому после смены
PyTorch, GPU или ОС может потребоваться новая серия в новых output/checkpoint
директориях копии config. Обычный одиночный `train.py` продолжает опыт после
перезапуска Colab, если сохранены код, данные и численные настройки.

Бесплатные вычислительные ресурсы Colab ограничены и могут прерываться:
https://research.google.com/colaboratory/faq.html


## Быстрая серия на одном seed с целевым бюджетом 55 минут

После подготовки проекта можно выполнить в отдельной ячейке:

```python
!python -u fast.py --seed 11 --minutes 55 --device cuda
```

Вся сетка k, все пары lr/lambda и полный train сохраняются, но предварительный
этап ограничен 8 эпохами, подтверждение — 30 эпохами и временем. Выбранные
модели используются для test без независимых финальных повторов. Сходимость
длительной серии не гарантируется. Seeds 22/33 запускаются отдельно.
Результаты и fast_status.json: results/svdpp-ml1m-fast/seed11/ на Drive.
Подробнее: docs/FAST_RU.md. Команда не выполнялась при подготовке проекта.
